**Python ile Derin Öğrenme Tabanlı Görüntü ve Doğal Dil İşleme – İSMEK**
Ödev: Model eğitimi
Ad Soyad: Ahmet ATAGÜL

Konu: Fotoğraftaki nesnenin neyden yapıldığını bulup içindeki tipik element oranlarını gösteren bir sistem yaptım. İki parçası var:
- kendi eğittiğim **malzeme modeli** (tuğla, seramik, kumaş, cam, deri, metal, kağıt, plastik, taş, ahşap...)
- hazır bir **nesne tanıma modeli** (telefon, kalem, kilit, şişe, kolye gibi 1000 nesneyi biliyor)

Nesneyi tanıyabilirse o nesneye özel bileşimi, tanıyamazsa malzemenin bileşimini gösteriyor. Böylece element çeşidi çok arttı.

### 1) GPU açık mı?
Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seçtim.

In [ ]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

# drive'ı en başta bağladım, yoksa eğitim bitene kadar izin penceresi bekliyor ve oturum kapanıyordu
from google.colab import drive
drive.mount('/content/drive')

### 2) Veri setini indirdim
MINC-2500 veri setini kullandım (Cornell Üniversitesi). 23 malzeme türü var, her birinde 2500 resim. Dosya büyük (yaklaşık 2,5 GB), inmesi birkaç dakika sürebiliyor.

Link çalışmazsa diye yedek olarak küçük TrashNet setini koydum.

In [ ]:
import os
os.makedirs('ham', exist_ok=True)

!wget -q --tries=2 --timeout=120 -O minc.tar.gz http://opensurfaces.cs.cornell.edu/static/minc/minc-2500.tar.gz

if os.path.exists('minc.tar.gz') and os.path.getsize('minc.tar.gz') > 100_000_000:
    print('MINC indi, açıyorum...')
    !tar -xzf minc.tar.gz -C ham
    kaynak = 'minc'
else:
    print('MINC inmedi, yedek olarak TrashNet kullanıyorum')
    !wget -q -O trashnet.zip https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip
    !unzip -q -o trashnet.zip -d ham
    kaynak = 'trashnet'

print('kaynak:', kaynak)

### 3) Kullanacağım sınıfları seçtim
MINC'te gökyüzü, su, ten, saç, yemek gibi nesne olmayan sınıflar da var. Bunları çıkardım, elementi belli olan 12 malzemeyi bıraktım.

İlk denemede her sınıftan 600 resim almıştım ve doğruluk %59 çıkmıştı. Model az resim gördüğü için zorlanıyordu, bu yüzden 2000'e çıkardım.

In [ ]:
import shutil, random

if kaynak == 'minc':
    secilen = ['brick', 'ceramic', 'fabric', 'foliage', 'glass', 'leather',
               'metal', 'paper', 'plastic', 'polishedstone', 'stone', 'wood']
    RESIM_SAYISI = 2000  # ilk denemede 600'dü
else:
    secilen = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
    RESIM_SAYISI = 10000  # trashnet küçük, hepsini al

# sınıf klasörlerinin olduğu yeri arıyorum (zip/tar içindeki yol farklı olabiliyor)
kok_klasor = None
for kok, klasorler, dosyalar in os.walk('ham'):
    if '__MACOSX' in kok:
        continue
    if 'glass' in klasorler and 'metal' in klasorler:
        kok_klasor = kok
        break
print('resimler burada:', kok_klasor)

random.seed(42)
veri_klasoru = 'veri'
shutil.rmtree(veri_klasoru, ignore_errors=True)
for sinif in secilen:
    dosyalar = [d for d in os.listdir(os.path.join(kok_klasor, sinif)) if d.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(dosyalar)
    os.makedirs(os.path.join(veri_klasoru, sinif))
    for d in dosyalar[:RESIM_SAYISI]:
        shutil.copy(os.path.join(kok_klasor, sinif, d), os.path.join(veri_klasoru, sinif, d))
    print(sinif, len(os.listdir(os.path.join(veri_klasoru, sinif))))

### 4) Birkaç resme baktım

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

plt.figure(figsize=(14, 8))
for i, sinif in enumerate(secilen):
    dosya = sorted(os.listdir(os.path.join(veri_klasoru, sinif)))[0]
    img = Image.open(os.path.join(veri_klasoru, sinif, dosya))
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(sinif)
    plt.axis('off')
plt.show()

### 5) Eğitim ve doğrulama olarak ayırdım
%80 eğitim, %20 doğrulama. Resimler 224x224.

Etiketleri one-hot (categorical) yaptım çünkü label smoothing kullanacağım, o bunu istiyor.

In [ ]:
BOYUT = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    veri_klasoru, validation_split=0.2, subset='training', seed=42,
    image_size=BOYUT, batch_size=BATCH, label_mode='categorical')

val_ds = tf.keras.utils.image_dataset_from_directory(
    veri_klasoru, validation_split=0.2, subset='validation', seed=42,
    image_size=BOYUT, batch_size=BATCH, label_mode='categorical')

sinif_adlari = train_ds.class_names
print(sinif_adlari)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

### 6) Malzeme modelini kurdum (transfer learning)
İlk denemede MobileNetV2 kullanmıştım. Bu sefer EfficientNetV2B0 kullandım, ImageNet'te daha başarılı ve hâlâ hızlı bir model. Alt katmanlarını dondurdum, üstüne kendi sınıflandırma katmanımı ekledim.

İlk denemeye göre değiştirdiklerim:
- augmentation'a dikey çevirme ve kontrast ekledim (malzemenin yönü önemli değil, ışık da değişebiliyor)
- label smoothing 0.1: model aşırı emin olmasın diye etiketleri biraz yumuşatıyor
- GPU'da mixed precision açtım, eğitim daha hızlı oluyor

In [ ]:
from tensorflow.keras import layers

if tf.config.list_physical_devices('GPU'):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),
])

# efficientnetv2 ön işlemeyi kendi içinde yapıyor, 0-255 resim veriyorum
base = tf.keras.applications.EfficientNetV2B0(input_shape=BOYUT + (3,), include_top=False, weights='imagenet')
base.trainable = False  # önce sadece üst katman eğitilsin

girdi = tf.keras.Input(shape=BOYUT + (3,))
x = augment(girdi)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
cikti = layers.Dense(len(sinif_adlari), activation='softmax', dtype='float32')(x)  # mixed precision'da son katman float32 olmalı

model = tf.keras.Model(girdi, cikti)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])
model.summary()

### 7) Eğittim
EarlyStopping ekledim: doğrulama 3 epoch boyunca iyileşmezse duruyor ve en iyi epoch'taki ağırlıklara geri dönüyor. ReduceLROnPlateau da ilerleme durunca learning rate'i yarıya indiriyor.

In [ ]:
erken_dur = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
lr_azalt = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1)

gecmis = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=[erken_dur, lr_azalt])

### 8) Fine-tuning
İlk denemede sadece son 30 katmanı açmıştım. Bu sefer son 100 katmanı açtım, ama BatchNormalization katmanlarını kapalı bıraktım (açınca eğitimi bozabiliyormuş). Learning rate küçük: 3e-5.

In [ ]:
base.trainable = True
for katman in base.layers[:-100]:
    katman.trainable = False
for katman in base.layers:
    if isinstance(katman, layers.BatchNormalization):
        katman.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(3e-5),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])

erken_dur2 = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
gecmis2 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[erken_dur2, lr_azalt])

### 9) Grafiklere baktım

In [ ]:
acc = gecmis.history['accuracy'] + gecmis2.history['accuracy']
val_acc = gecmis.history['val_accuracy'] + gecmis2.history['val_accuracy']
loss = gecmis.history['loss'] + gecmis2.history['loss']
val_loss = gecmis.history['val_loss'] + gecmis2.history['val_loss']
ft = len(gecmis.history['accuracy']) - 1  # fine-tuning burada başladı

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='eğitim')
plt.plot(val_acc, label='doğrulama')
plt.axvline(ft, color='gray', linestyle='--')
plt.title('accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='eğitim')
plt.plot(val_loss, label='doğrulama')
plt.axvline(ft, color='gray', linestyle='--')
plt.title('loss')
plt.legend()
plt.show()

### 10) Hangi malzemeleri karıştırıyor?

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

gercek, tahmin = [], []
for resimler, etiketler in val_ds:
    olasilik = model.predict(resimler, verbose=0)
    tahmin.extend(np.argmax(olasilik, axis=1))
    gercek.extend(np.argmax(etiketler.numpy(), axis=1))

etiket_no = list(range(len(sinif_adlari)))
print(classification_report(gercek, tahmin, labels=etiket_no, target_names=sinif_adlari, zero_division=0))

cm = confusion_matrix(gercek, tahmin, labels=etiket_no)
fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(cm, display_labels=sinif_adlari).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.show()

İlk denemede (MobileNetV2, sınıf başına 600 resim) doğruluk %59 çıkmıştı. En iyi taş, deri ve tuğlayı tanıyordu, en çok metal, cilalı taş, plastik ve seramikte zorlanıyordu. Bunların yüzeyi pürüzsüz ve parlak, birbirine benziyor.

Değişikliklerden sonra doğruluk %79 oldu. En çok deriyi kumaşla (61 resim), tuğlayı taşla (49 resim) karıştırıyor. En düşük skor metal (0.68) ve plastikte (0.74).

### 11) Malzemelerin element tablosu
Fotoğraftan element ölçülemez, bunun için XRF gibi cihaz lazım. O yüzden her malzemenin tipik bileşimini (kütlece yaklaşık %) bir sözlüğe yazdım.

Nasıl hesapladım:
- cam, seramik, tuğla, granit: içlerindeki oksitlerin oranından (ör. SiO2, Al2O3, CaO) elementlere çevirdim
- kağıt, pamuklu kumaş: selüloz (C6H10O5)n, kağıtta biraz kalsiyum karbonat dolgu da var
- taş: kireçtaşı varsaydım (CaCO3)
- deri: kollajen proteini, krom ile tabaklandığı için içinde biraz Cr var
- plastik: en yaygını polietilen/polipropilen (CH2)n
- metal: sıradan çelik varsaydım
- yaprak: kuru ağırlık üzerinden

In [ ]:
# yaklaşık kütlece yüzdeler
malzeme_bilesim = {
    'brick':         {'O': 48, 'Si': 28, 'Al': 9, 'Fe': 5, 'Ca': 3, 'K': 2, 'Mg': 1.5, 'Na': 1, 'Ti': 0.5, 'diğer': 2},
    'ceramic':       {'O': 50.1, 'Si': 32.7, 'Al': 13.2, 'K': 2.9, 'Ca': 1.1},
    'fabric':        {'O': 49.4, 'C': 44.4, 'H': 6.2},
    'foliage':       {'C': 45, 'O': 42, 'H': 6, 'N': 2.5, 'K': 1.5, 'Ca': 1.5, 'Mg': 0.3, 'P': 0.3, 'S': 0.3, 'diğer': 0.6},
    'glass':         {'O': 46.6, 'Si': 33.7, 'Na': 10.4, 'Ca': 6.4, 'Mg': 2.4, 'Al': 0.5},
    'leather':       {'C': 47, 'O': 26, 'N': 16, 'H': 6.5, 'Cr': 2.5, 'S': 0.5, 'diğer': 1.5},
    'metal':         {'Fe': 98, 'Mn': 0.8, 'Si': 0.3, 'C': 0.2, 'diğer': 0.7},
    'paper':         {'O': 49, 'C': 41, 'H': 5.8, 'Ca': 2.5, 'diğer': 1.7},
    'plastic':       {'C': 85.7, 'H': 14.3},
    'polishedstone': {'O': 48.5, 'Si': 33.5, 'Al': 7.5, 'K': 3.5, 'Na': 2.8, 'Fe': 2, 'Ca': 1.3, 'Mg': 0.4, 'Ti': 0.2, 'diğer': 0.3},
    'stone':         {'O': 48, 'Ca': 40, 'C': 12},
    'wood':          {'C': 50, 'O': 43, 'H': 6, 'N': 0.2, 'diğer': 0.8},
    # yedek veri seti (trashnet) için
    'cardboard':     {'O': 49.4, 'C': 44.4, 'H': 6.2},
    'trash':         {},
}

malzeme_tr = {
    'brick': 'tuğla', 'ceramic': 'seramik', 'fabric': 'kumaş (pamuk)', 'foliage': 'yaprak/bitki',
    'glass': 'cam', 'leather': 'deri', 'metal': 'metal (çelik)', 'paper': 'kağıt',
    'plastic': 'plastik (PE/PP)', 'polishedstone': 'cilalı taş (granit)', 'stone': 'taş (kireçtaşı)',
    'wood': 'ahşap', 'cardboard': 'karton', 'trash': 'karışık çöp',
}

### 12) Nesne katmanı
Malzeme modeli "metal" diyor ama metal çelik de olabilir, bakır da, altın da. Burada hazır MobileNetV2'nin 1000 sınıflı tam halini kullandım (bunu ben eğitmedim). Nesneyi tanıyınca o nesneye özel bileşimi gösteriyorum.

Bazı varsayımlar yaptım: kolyeyi 14 ayar altın, asma kilidi pirinç, telefonu ortalama bir akıllı telefon kabul ettim. Telefon ve laptop değerleri kabaca, modelden modele çok değişiyor.

In [ ]:
cam = malzeme_bilesim['glass']
seramik = malzeme_bilesim['ceramic']
pet = {'C': 62.5, 'O': 33.3, 'H': 4.2}
celik = {'Fe': 98.5, 'Mn': 0.8, 'Zn': 0.5, 'C': 0.2}  # galvanizli
pirinc = {'Cu': 65, 'Zn': 33, 'Pb': 1.5, 'diğer': 0.5}

# imagenet sınıf adı: (türkçe ad, bileşim)
nesne_bilesim = {
    'ballpoint':          ('tükenmez kalem', {'C': 80, 'H': 13, 'Cu': 3, 'Zn': 2, 'W': 1, 'diğer': 1}),
    'fountain_pen':       ('dolma kalem', {'C': 70, 'H': 11, 'Fe': 10, 'O': 5, 'Cr': 2.5, 'Ni': 1, 'diğer': 0.5}),
    'rubber_eraser':      ('silgi', {'C': 50, 'Cl': 35, 'H': 7, 'O': 5, 'Ca': 3}),
    'pencil_sharpener':   ('kalemtıraş', {'Mg': 90, 'Al': 6, 'Zn': 1, 'Fe': 3}),
    'beer_bottle':        ('cam şişe', cam),
    'wine_bottle':        ('şarap şişesi', cam),
    'beer_glass':         ('bardak', cam),
    'water_bottle':       ('pet şişe', pet),
    'pop_bottle':         ('gazoz şişesi (PET)', pet),
    'coffee_mug':         ('kupa', seramik),
    'teapot':             ('demlik', seramik),
    'plate':              ('tabak', seramik),
    'vase':               ('vazo', seramik),
    'frying_pan':         ('tava (teflon kaplı alüminyum)', {'Al': 92, 'F': 5.5, 'C': 1.5, 'Mg': 1}),
    'wok':                ('wok tava (karbon çeliği)', {'Fe': 99, 'C': 0.5, 'Mn': 0.5}),
    'chain':              ('zincir', celik),
    'nail':               ('çivi', celik),
    'screw':              ('vida', celik),
    'safety_pin':         ('çengelli iğne (paslanmaz çelik)', {'Fe': 72, 'Cr': 18, 'Ni': 8, 'Mn': 1.5, 'Si': 0.5}),
    'hammer':             ('çekiç', {'Fe': 75, 'C': 13, 'O': 10, 'H': 1.5, 'Mn': 0.5}),
    'padlock':            ('asma kilit (pirinç)', pirinc),
    'cornet':             ('kornet (pirinç)', {'Cu': 70, 'Zn': 30}),
    'trombone':           ('trombon (pirinç)', {'Cu': 70, 'Zn': 30}),
    'French_horn':        ('korno (pirinç)', {'Cu': 70, 'Zn': 30}),
    'necklace':           ('kolye (14 ayar altın varsaydım)', {'Au': 58.5, 'Cu': 30, 'Ag': 9, 'Zn': 2.5}),
    'cellular_telephone': ('cep telefonu', {'O': 22, 'C': 20, 'Al': 15, 'Si': 11, 'Cu': 10, 'Co': 5, 'Fe': 5, 'H': 3,
                                            'Ni': 1.5, 'Li': 0.7, 'Sn': 0.6, 'Mg': 0.5, 'Ag': 0.1, 'Au': 0.03, 'diğer': 5.57}),
    'laptop':             ('dizüstü bilgisayar', {'C': 25, 'O': 20, 'Al': 18, 'Si': 10, 'Cu': 8, 'Fe': 8, 'H': 3, 'Co': 3,
                                                  'Ni': 1, 'Mg': 1, 'Sn': 0.5, 'Li': 0.4, 'Ag': 0.05, 'Au': 0.01, 'diğer': 2.04}),
    'wooden_spoon':       ('tahta kaşık', malzeme_bilesim['wood']),
    'candle':             ('mum (parafin)', {'C': 85, 'H': 15}),
    'tennis_ball':        ('tenis topu', {'C': 70, 'H': 9, 'O': 8, 'Si': 5, 'Zn': 2, 'S': 2, 'N': 2, 'diğer': 2}),
    'golf_ball':          ('golf topu', {'C': 80, 'H': 12, 'O': 3, 'Zn': 3, 'diğer': 2}),
    'running_shoe':       ('spor ayakkabı', {'C': 70, 'O': 15, 'H': 10, 'N': 3, 'diğer': 2}),
    'carton':             ('karton kutu', malzeme_bilesim['cardboard']),
    'envelope':           ('zarf', malzeme_bilesim['paper']),
    # tornavida: krom-vanadyum çelik uç (~%55) + plastik sap (~%45) varsaydım
    'screwdriver':        ('tornavida', {'Fe': 53.4, 'C': 38.9, 'H': 6.4, 'Cr': 0.55, 'Mn': 0.45, 'Si': 0.17, 'V': 0.11, 'diğer': 0.02}),
    'cup':                ('fincan', seramik),
}
# nesne modeli telefonu bazen bu adlarla görüyor
nesne_bilesim['hand-held_computer'] = nesne_bilesim['cellular_telephone']
nesne_bilesim['iPod'] = nesne_bilesim['cellular_telephone']

element_tr = {
    'H': 'hidrojen', 'Li': 'lityum', 'C': 'karbon', 'N': 'azot', 'O': 'oksijen', 'F': 'flor',
    'Na': 'sodyum', 'Mg': 'magnezyum', 'Al': 'alüminyum', 'Si': 'silisyum', 'P': 'fosfor',
    'S': 'kükürt', 'Cl': 'klor', 'K': 'potasyum', 'Ca': 'kalsiyum', 'Ti': 'titanyum',
    'Cr': 'krom', 'Mn': 'mangan', 'Fe': 'demir', 'Co': 'kobalt', 'Ni': 'nikel', 'Cu': 'bakır',
    'Zn': 'çinko', 'Ag': 'gümüş', 'Sn': 'kalay', 'W': 'tungsten', 'Au': 'altın', 'Pb': 'kurşun', 'V': 'vanadyum',
    'diğer': 'diğer',
}

# kontrol: toplamlar 100 mü, hepsinin türkçe adı var mı
tum_tablolar = list(malzeme_bilesim.items()) + [(k, v[1]) for k, v in nesne_bilesim.items()]
for ad, b in tum_tablolar:
    if b and abs(sum(b.values()) - 100) > 0.5:
        print('TOPLAM HATALI:', ad, sum(b.values()))
    for e in b:
        if e not in element_tr:
            print('türkçesi yok:', e)

farkli = sorted({e for _, b in tum_tablolar for e in b if e != 'diğer'})
print(len(farkli), 'farklı element:', ', '.join(farkli))

In [ ]:
nesne_modeli = tf.keras.applications.MobileNetV2(weights='imagenet')  # 1000 sınıflı hazır model

def nesne_bul(img, kac=3):
    dizi = np.array(img.convert('RGB').resize(BOYUT), dtype='float32')[None, ...]
    dizi = tf.keras.applications.mobilenet_v2.preprocess_input(dizi)
    sonuc = tf.keras.applications.mobilenet_v2.decode_predictions(nesne_modeli.predict(dizi, verbose=0), top=kac)[0]
    return [(ad, float(p)) for _, ad, p in sonuc]

def malzeme_bul(img):
    dizi = np.array(img.convert('RGB').resize(BOYUT), dtype='float32')[None, ...]  # ön işleme modelin içinde
    olasilik = model.predict(dizi, verbose=0)[0]
    i = int(np.argmax(olasilik))
    return sinif_adlari[i], float(olasilik[i])

### 13) Sonucu Türkçe cümleye çeviren fonksiyon
Önce nesneye bakıyor. İlk 3 tahminden biri benim tablomda varsa ve olasılığı %20'den büyükse onu kullanıyor. Yoksa malzeme modelinin sonucunu kullanıyor.

In [ ]:
def element_metni(bilesim):
    sirali = sorted(bilesim.items(), key=lambda t: -t[1])
    return ', '.join(f"%{v:g} {element_tr[k]} ({k})" if k != 'diğer' else f"%{v:g} diğer" for k, v in sirali)

def acikla(img, goster=True):
    nesneler = nesne_bul(img)
    malzeme, m_guven = malzeme_bul(img)

    secilen_nesne = None
    for ad, p in nesneler:
        if ad in nesne_bilesim and p > 0.2:
            secilen_nesne = (ad, p)
            break

    if secilen_nesne:
        ad, p = secilen_nesne
        tr_ad, bilesim = nesne_bilesim[ad]
        cumle = f"Bu büyük olasılıkla bir {tr_ad} (%{p*100:.0f}). "
        cumle += f"Malzeme modelim yüzeyi {malzeme_tr[malzeme]} olarak gördü (%{m_guven*100:.0f}). "
    else:
        bilesim = malzeme_bilesim[malzeme]
        if m_guven < 0.5:
            cumle = f"Nesneyi tanıyamadım, malzemeden de tam emin değilim. En yakın tahminim {malzeme_tr[malzeme]} (%{m_guven*100:.0f}). "
        else:
            cumle = f"Nesneyi tanıyamadım ama malzemesi büyük olasılıkla {malzeme_tr[malzeme]} (%{m_guven*100:.0f}). "

    if bilesim:
        cumle += f"İçinde {len([e for e in bilesim if e != 'diğer'])} farklı element var: {element_metni(bilesim)}."
    else:
        cumle += "Karışık olduğu için belli bir element oranı veremiyorum."
    cumle += " (Ölçüm değil, tipik bileşimdir.)"

    if goster:
        plt.imshow(img)
        plt.axis('off')
        plt.show()
        print('nesne modelinin ilk 3 tahmini:', [(a, round(p, 2)) for a, p in nesneler])
    print(cumle)
    return cumle

In [ ]:
# önce veri setinden birkaç resimle denedim
for sinif in secilen[:3]:
    dosya = sorted(os.listdir(os.path.join(veri_klasoru, sinif)))[3]
    acikla(Image.open(os.path.join(veri_klasoru, sinif, dosya)))

### 14) Kendi çektiğim fotoğraflarla denedim
Telefon, kalem, kupa, anahtar gibi evdeki şeylerin fotoğrafını çekip yükledim.

In [ ]:
from google.colab import files
yuklenen = files.upload()
for ad in yuklenen:
    acikla(Image.open(ad))

### 15) Modeli kaydettim

In [ ]:
import json
klasor = '/content/drive/MyDrive/malzeme_projesi'
os.makedirs(klasor, exist_ok=True)
model.save(klasor + '/malzeme_modeli.keras')  # drive'a kaydettim, bir daha eğitmeye gerek yok
json.dump(sinif_adlari, open(klasor + '/siniflar.json', 'w'))
print('kaydedildi:', klasor)

### 16) Arayüz
Sunumda göstermek için Gradio ile resim yükleme ekranı yaptım. Çalıştırınca bir link veriyor.

In [ ]:
!pip install -q gradio
import gradio as gr

def arayuz(img):
    dizi = np.array(img.convert('RGB').resize(BOYUT), dtype='float32')[None, ...]
    olasilik = model.predict(dizi, verbose=0)[0]
    malzemeler = {malzeme_tr[s]: float(o) for s, o in zip(sinif_adlari, olasilik)}
    return malzemeler, acikla(img, goster=False)

gr.Interface(fn=arayuz, inputs=gr.Image(type='pil'),
             outputs=[gr.Label(num_top_classes=3, label='malzeme'), gr.Textbox(label='açıklama')],
             title='Nesne, malzeme ve element tahmini').launch(share=True)

### 17) Ne yaptım?
İlk planım 6 sınıflı TrashNet ile çalışmaktı ama orada çıkan elementler hep aynı oluyordu (C, H, O, Si, Al...). Element çeşidini artırmak için iki şey yaptım. Önce veri setini 23 malzemeli MINC-2500 ile değiştirdim ve içinden 12 malzeme seçip transfer learning ve fine-tuning yaptım. İlk denemede MobileNetV2 ve sınıf başına 600 resimle %59 aldım. Sonra EfficientNetV2B0'a geçtim, resim sayısını 2000'e çıkardım, augmentation'ı artırdım, label smoothing ve EarlyStopping ekledim, fine-tuning'de daha çok katman açtım. Tuğla, granit, deri gibi sınıflar gelince Fe, Ti, K, Cr gibi elementler de tabloya girdi.

Sonra malzeme modelinin yetmediği yeri gördüm: "metal" deyince çelik mi, bakır mı, altın mı bilemiyor. Bunun için hazır ImageNet modelini nesne tanımak için kullandım ve 30'dan fazla nesneye özel bileşim yazdım. Telefon gelince kobalt, lityum, altın, gümüş; pirinç kilit gelince bakır, çinko; kolye gelince altın çıkıyor.

Sistem element ölçmüyor, sadece tanıdığı nesne ya da malzemenin tipik bileşimini gösteriyor. Varsayımlarım da var (kolye altın, plastik PE gibi). Gerçek ölçüm için XRF gibi bir cihaz gerekiyor.